In [1]:
!pip install --quiet scikit-learn

In [2]:
%cd ..

/Users/danorel/Workspace/Education/University/NYU/Research/xeda


In [3]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv('.env'))

True

In [4]:
import random
import chromadb
import copy
import json
import typing as t
import pandas as pd
import numpy as np
import uuid
import pathlib
import s3fs
import openai
import traceback

from dagster import EnvVar
from sklearn.metrics.pairwise import cosine_similarity
from time import time
from tqdm import tqdm

from pipeline.resources import S3FSResource
from pipeline.solid.pipeline_annotator import annotate_pipeline
from pipeline.solid.utils.model_manager import ModelManager
from pipeline.solid.utils.pipelines.pipeline_precalculated_sets import PipelineWithPrecalculatedSets
from typings.pipeline import OperatorRequestData


from constants import (
    AWS_ACCESS_KEY_ID,
    AWS_SECRET_ACCESS_KEY,
    AWS_S3_ENDPOINT_URL,
    AWS_S3_REGION_NAME,
    AWS_S3_BUCKET_NAME,
    AWS_S3_USE_SSL,
    GROUPS_CSV_PATH,
    OPENAI_API_KEY,
    OPENAI_EMBEDDINGS_MODEL,
    VECTOR_STORE_COLLECTION,
    VECTOR_STORE_HOST,
    VECTOR_STORE_PORT,
    UNIVERSAL_POLICY_NAME
)
from typings.pipeline import Pipeline, PipelineEda4Sum
from pipeline.solid.pipeline_sampler import next_pipeline_iter
from utils.s3 import pull_keras_model
from utils.vector_store import ChromaDBVectorStore, MilvusVectorStore, make_document_sampler

2024-05-02 01:54:27.589668: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Preparing dataset

In [5]:
embedding_client = openai.OpenAI(api_key=OPENAI_API_KEY)

In [6]:
vector_store = MilvusVectorStore(
    host=VECTOR_STORE_HOST, 
    port=VECTOR_STORE_PORT,
    collection_name=VECTOR_STORE_COLLECTION
)

In [7]:
fs = s3fs.S3FileSystem(
    key=AWS_ACCESS_KEY_ID,
    secret=AWS_SECRET_ACCESS_KEY,
    endpoint_url=AWS_S3_ENDPOINT_URL,
    use_ssl=AWS_S3_USE_SSL,
    client_kwargs={"region_name": AWS_S3_REGION_NAME},
)

In [8]:
database_pipeline_cache = {}
database_pipeline_cache["galaxies"] = PipelineWithPrecalculatedSets(
    "sdss",
    ["galaxies"],
    discrete_categories_count=10,
    min_set_size=10,
    exploration_columns=[
        "galaxies.u",
        "galaxies.g",
        "galaxies.r",
        "galaxies.i",
        "galaxies.z",
        "galaxies.petroRad_r",
        "galaxies.redshift",
    ],
    id_column="galaxies.objID",
)

In [9]:
model_manager = ModelManager(database_pipeline_cache["galaxies"], models = {
    "set": pull_keras_model(
        s3fs=fs,
        bucket_name=AWS_S3_BUCKET_NAME,
        policy_name=UNIVERSAL_POLICY_NAME,
        model_name="set_actor",
    ),
    "operation": pull_keras_model(
        s3fs=fs,
        bucket_name=AWS_S3_BUCKET_NAME,
        policy_name=UNIVERSAL_POLICY_NAME,
        model_name="operation_actor",
    ),
    "set_op_counters": None,
})

In [10]:
groups_df = pd.read_csv(GROUPS_CSV_PATH)

In [11]:
def make_embedding(text):
    response = embedding_client.embeddings.create(
        input=text,
        model=OPENAI_EMBEDDINGS_MODEL
    )
    return response.data[0].embedding

In [12]:
def node_to_annotation_encoding(node):
    annotation = node["annotation"]
    node_encoding = []
    for k, v in annotation.items():
        if isinstance(v, dict):
            for key in v:
                node_encoding.append(f"{k}_{key} = {v[key]}")
        else:
            node_encoding.append(f"{k} = {v}")
    return ', '.join(node_encoding)


def annotation_subset_to_embedding(annotation_subset: t.List[str]):
    annotation_text = ';'.join(annotation_subset)
    annotation_embedding = make_embedding(annotation_text)
    return annotation_embedding


def pipeline_to_annotation_subsets(pipeline: Pipeline) -> t.List[Pipeline]:
    annotation_subsets = []
    partial_annotation = []
    for node in reversed(pipeline):
        encoded_annotation = node_to_annotation_encoding(node)
        partial_annotation.append(encoded_annotation)
        annotation_subsets.append(copy.deepcopy(partial_annotation))
    return annotation_subsets


def pipeline_to_annotation_payloads(pipeline: Pipeline):
    annotation_subsets = pipeline_to_annotation_subsets(pipeline)
    annotation_payloads = (
        [str(uuid.uuid4()) for _ in range(len(annotation_subsets))],
        [annotation_subset_to_embedding(annotation_subset) for annotation_subset in annotation_subsets]
    )
    return annotation_payloads


def pipeline_to_encoding(annotated_pipeline: PipelineEda4Sum):
    return ';'.join([node_to_annotation_encoding(node) for node in annotated_pipeline])


def pipeline_to_embedding(pipeline: PipelineEda4Sum):
    annotated_pipeline = annotate_pipeline(groups_df, pipeline)
    pipeline_encoding = pipeline_to_encoding(annotated_pipeline)
    pipeline_embedding = make_embedding(pipeline_encoding)
    return pipeline_embedding

In [ ]:
def fetch_pipelines():
    for annotated_file in fs.glob('xeda/annotated_pipelines/*.json'):
        with fs.open(annotated_file, 'r') as f:
            annotated_pipeline = json.load(f)
            if not isinstance(annotated_pipeline, list) or not len(annotated_pipeline):
                continue
            serialized_pipeline = json.dumps(annotated_pipeline)
            if len(serialized_pipeline) > 65535:
                continue
            yield annotated_pipeline, serialized_pipeline


def insert_pipelines(vector_store, refresh: bool = True):
    if not refresh:
        print("Skipping refresh of vector storage")
        return
    for annotated_pipeline, serialized_pipeline in tqdm(fetch_pipelines()):
        (
            annotation_ids,
            annotation_embeddings
        ) = pipeline_to_annotation_payloads(annotated_pipeline)
        if len(annotation_ids) > 0:
            vector_store.insert(
                ids=annotation_ids,
                documents=[serialized_pipeline for _ in range(len(annotation_ids))],
                embeddings=annotation_embeddings,
            )

insert_pipelines(vector_store, refresh=True)

30it [01:07,  2.01s/it]

### Experiments

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def explore_pipeline(partial_pipeline: Pipeline, database_pipeline_cache, k: int):
    # This is a stub to simulate pipeline exploration.
    # In a real scenario, this would involve applying transformations or decisions based on an RL model.

    partial_latest_node = partial_pipeline[-1]
    partial_latest_request_data = partial_latest_node.get("requestData")
    
    terminal_request_data = OperatorRequestData(**partial_latest_request_data)
    terminal_pipeline = partial_pipeline.copy()
    
    for i in range(len(partial_pipeline), len(partial_pipeline) + k):
        try:
            start_time = time()
            terminal_node, terminal_request_data = next_pipeline_iter(
                database_pipeline_cache,
                model_manager,
                terminal_request_data
            )
            print(f"Made step in {time() - start_time}s")
            terminal_pipeline.append(terminal_node)
        except ValueError:
            print(f"Unexpectedly exited from pipeline generation on step {i}. Saving pipeline as it is...")
            break
    
    return annotate_pipeline(groups_df, terminal_pipeline)


def similarity(emb1, emb2, type_of_similarity):
    if type_of_similarity == 'cosine':
        return np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    elif type_of_similarity == 'euclidian':
        return np.linalg.norm(emb1 - emb2)
    elif type_of_similarity == 'manhattan':
        return np.abs(emb1 - emb2).sum()
    else:
        raise NotImplementedError("Unknown type of similarity")


def result_to_pipeline(result: dict):
    try:
        return json.loads(result.get('document'))
    except:
        print(result.get('id'), result.get('document'))
        return {}


def compute_embedding(pipeline):
    embedding = pipeline_to_embedding(pipeline)
    return np.array(embedding)


def make_experiment(vector_store, partial_pipeline, partial_pipeline_id, type_of_similarity: str, k: int):
    print(f"Starting an experiment with k='{k}' exploration steps and type_of_similarity='{type_of_similarity}'")
    start_time = time()

    partial_embedding = compute_embedding(partial_pipeline)
    
    terminal_pipeline = explore_pipeline(partial_pipeline, database_pipeline_cache, k=k)
    terminal_embedding = compute_embedding(terminal_pipeline)
    
    partial_search_results = vector_store.search(embedding=partial_embedding, k=1000)

    min_pipeline, max_pipeline = result_to_pipeline(partial_search_results[0]), result_to_pipeline(partial_search_results[-1])
    min_embedding = compute_embedding(min_pipeline)
    max_embedding = compute_embedding(max_pipeline)

    min_to_terminal_similarity = similarity(terminal_embedding, min_embedding, type_of_similarity)
    max_to_terminal_similarity = similarity(terminal_embedding, max_embedding, type_of_similarity)

    print(f"Finished an experiment with k='{k}' exploration steps and type_of_similarity='{type_of_similarity}' in {time() - start_time}s")
    
    return {
        "partial_pipeline_id": partial_pipeline_id,
        "min_to_terminal_similarity": min_to_terminal_similarity,
        "max_to_terminal_similarity": max_to_terminal_similarity,
    }


def experiment_metadata(experiment, type_of_similarity, k):
    r = {
        "experiment": experiment + 1,
        "exploration_steps": k,
        "type_of_similarity": type_of_similarity
    }
    return r


def run_single_experiment(args):
    try:
        experiment, vector_store, partial_pipeline, partial_pipeline_id, types_of_similarity, exploration_steps = args
        results = []
        for k in range(*exploration_steps):
            for type_of_similarity in types_of_similarity:
                experiment_config = (type_of_similarity, k)
                results.append({
                    **experiment_metadata(
                        experiment,
                        *experiment_config
                    ),
                    **make_experiment(
                        vector_store,
                        partial_pipeline,
                        partial_pipeline_id,
                        *experiment_config
                    )
                })
        return results
    except Exception as e:
        print(f"Error in experiment {args[0]}: {e}")  # Log the error and experiment details
        print(traceback.format_exc())
        return []  # Return an empty list or some error indicator


def run_experiments(n: int = 1000, types_of_similarity: list = ['cosine'], exploration_steps: tuple = (3, 6), max_workers: int = 1):
    pipeline_sampler = make_document_sampler(vector_store)

    # Prepare arguments for each experiment
    experiments_args = []
    for experiment in range(n):
        sampled_pipeline = None
        while sampled_pipeline is None:
            sampled_pipeline_index, sampled_pipeline_id = pipeline_sampler()
            try:
                sampled_pipeline = result_to_pipeline(vector_store.get(sampled_pipeline_id))
            except Exception as e:
                print(f"Failed to parse Pipeline[id={sampled_pipeline_id}]")
        experiments_args.append((experiment, vector_store, sampled_pipeline, sampled_pipeline_id, types_of_similarity, exploration_steps))
    
    # Run experiments in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(executor.map(run_single_experiment, experiments_args), total=n))

    # Collect all results into a single DataFrame
    experiments = pd.DataFrame([item for sublist in results for item in sublist])
    return experiments

In [ ]:
experiments_path = 'data/experiments.csv'

if pathlib.Path(experiments_path).exists():
    experiments_df = pd.read_csv(experiments_path)
else:
    experiments_df = run_experiments(
        n=50,
        types_of_similarity=['manhattan', 'cosine', 'euclidian'],
        exploration_steps=(1, 8)
    )

In [ ]:
experiments_success_condition = (experiments_df['max_to_terminal_similarity'] < experiments_df['min_to_terminal_similarity'])
experiments_df['explanation_satisfied'] = experiments_success_condition
experiments_df.to_csv(experiments_path, index=False)

### Visualizations

In [ ]:
!pip install --quiet matplotlib seaborn

In [ ]:
import matplotlib.pyplot as plt
import math
import seaborn as sns

In [ ]:
visualization_df = experiments_df.copy()

In [ ]:
visualization_df

In [ ]:
cosine_3k_df = visualization_df.loc[
    (visualization_df['type_of_similarity'] == 'cosine') & 
    (visualization_df['exploration_steps'] == 3)
][['experiment', 'min_to_terminal_similarity', 'max_to_terminal_similarity']]

plot = cosine_3k_df.plot(
    kind='bar',
    x='experiment', 
    y=['min_to_terminal_similarity', 'max_to_terminal_similarity'],
    title='Metric: cosine; Exploration steps = 3',
    figsize=(12, 5)
)
plt.show()

In [ ]:
euclidian_3k_df = visualization_df.loc[
    (visualization_df['type_of_similarity'] == 'euclidian') & 
    (visualization_df['exploration_steps'] == 3)
][['experiment', 'min_to_terminal_similarity', 'max_to_terminal_similarity']]

plot = euclidian_3k_df.plot(
    kind='bar', 
    x='experiment', 
    y=['min_to_terminal_similarity', 'max_to_terminal_similarity'],
    title='Metric: euclidian; Exploration steps = 3',
    figsize=(12, 5)
)
plt.show()

In [ ]:
manhattan_3k_df = visualization_df.loc[
    (visualization_df['type_of_similarity'] == 'manhattan') & 
    (visualization_df['exploration_steps'] == 3)
][['experiment', 'min_to_terminal_similarity', 'max_to_terminal_similarity']]

plot = manhattan_3k_df.plot(
    kind='bar', 
    x='experiment', 
    y=['min_to_terminal_similarity', 'max_to_terminal_similarity'],
    title='Metric: manhattan; Exploration steps = 3',
    figsize=(12, 5)
)
plt.show()

In [ ]:
k_values = visualization_df['exploration_steps'].unique()

In [ ]:
for k in k_values:
    euclidian_df = visualization_df.loc[
        (visualization_df['type_of_similarity'] == 'euclidian') & 
        (visualization_df['exploration_steps'] == k)
    ]
    amount_of_success_experiments = len(euclidian_df.loc[euclidian_df['explanation_satisfied'] == True])
    amount_of_total_experiments = len(set(euclidian_df['experiment'].values))
    euclidian_success_percentage = round((amount_of_success_experiments / amount_of_total_experiments) * 100, 2)
    print(f"Euclidian success cases for {k} exploration steps: {euclidian_success_percentage}%")

In [ ]:
for k in k_values:
    cosine_df = visualization_df.loc[
        (visualization_df['type_of_similarity'] == 'cosine') & 
        (visualization_df['exploration_steps'] == k)
    ]
    amount_of_success_experiments = len(cosine_df.loc[cosine_df['explanation_satisfied'] == True])
    amount_of_total_experiments = len(set(cosine_df['experiment'].values))
    cosine_success_percentage = round((amount_of_success_experiments / amount_of_total_experiments) * 100, 2)
    print(f"Cosine success cases for {k} exploration steps: {cosine_success_percentage}%")

In [ ]:
for k in k_values:
    manhattan_df = visualization_df.loc[
        (visualization_df['type_of_similarity'] == 'manhattan') & 
        (visualization_df['exploration_steps'] == k)
    ]
    amount_of_success_experiments = len(manhattan_df.loc[manhattan_df['explanation_satisfied'] == True])
    amount_of_total_experiments = len(set(manhattan_df['experiment'].values))
    manhattan_success_percentage = round((amount_of_success_experiments / amount_of_total_experiments) * 100, 2)
    print(f"Manhattan success cases for {k} exploration steps: {manhattan_success_percentage}%")

### Plots

In [ ]:
# Grouping data by type_of_similarity and k, and calculate success rates
success_rates = experiments_df.groupby(['type_of_similarity', 'exploration_steps']).agg(
    total_experiments=('explanation_satisfied', 'count'),
    successful_experiments=('explanation_satisfied', 'sum')
).reset_index()

success_rates['success_rate'] = (success_rates['successful_experiments'] / success_rates['total_experiments']) * 100

# Displaying the calculated success rates
success_rates

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Setting up the plot
plt.figure(figsize=(12, 6))
sns.barplot(data=success_rates, x='type_of_similarity', y='success_rate', hue='exploration_steps')
plt.title('Success Rate by Type of Similarity and Number of Exploration Steps (k)')
plt.xlabel('Type of Similarity')
plt.ylabel('Success Rate (%)')
plt.legend(title='Exploration Steps')
plt.show()

In [ ]:
import numpy as np

# Re-calculating the Coefficient of Variation (CV) for each type of similarity
cv_data = success_rates.groupby('type_of_similarity')['success_rate'].apply(
    lambda x: np.std(x) / np.mean(x) * 100  # CV as a percentage
).reset_index().rename(columns={'success_rate': 'Coefficient_of_Variation'})

# Plotting the Coefficient of Variation
plt.figure(figsize=(10, 6))
sns.barplot(data=cv_data, x='type_of_similarity', y='Coefficient_of_Variation', palette='viridis')
plt.title('Coefficient of Variation (CV) for Success Rates by Similarity Type')
plt.xlabel('Type of Similarity')
plt.ylabel('Coefficient of Variation (%)')
plt.show()

### Best configuration

In [ ]:
# Grouping the data by type of similarity and k, summarizing success rates
success_summary = experiments_df.groupby(['type_of_similarity', 'exploration_steps']).agg(
    mean_success_rate=('explanation_satisfied', lambda x: x.mean() * 100)
).reset_index()

# Plotting the success rate trends for each type of similarity across different k
plt.figure(figsize=(12, 8))
sns.lineplot(data=success_summary, x='exploration_steps', y='mean_success_rate', hue='type_of_similarity', marker='o')
plt.title('Success Rate Trends by Type of Similarity and Exploration Steps (k)')
plt.xlabel('Number of Exploration Steps (k)')
plt.ylabel('Mean Success Rate (%)')
plt.legend(title='Type of Similarity')
plt.grid(True)
plt.show()

In [ ]:
experiments_df['similarity_difference'] = experiments_df['max_to_terminal_similarity'] - experiments_df['min_to_terminal_similarity']

plt.figure(figsize=(18, 8))
sns.scatterplot(data=experiments_df, x='experiment', y='similarity_difference', hue='type_of_similarity', style='type_of_similarity', palette='Set1', s=50)
plt.title('Difference Between Max and Min Terminal Similarities Across Experiments')
plt.xlabel('Experiment Number')
plt.ylabel('Difference (Max - Min Terminal Similarity)')
plt.legend(title='Type of Similarity')
plt.grid(True)
plt.show()